# 01 · Ingesta Bronze desde cero

Descarga los originales de Ookla, OSIPTEL, INEI y WorldPop y los publica bajo `run_id=…/<fuente>/`.
Cada objeto queda asociado a su origen y a su SHA-256. Al completar los controles, la notebook publica el manifiesto
que consume Silver. Ejecutar en el nodo primario de EMR.

## 1. Configuración

In [1]:
import pandas as pd
from IPython.display import display


def show_values(values):
    show_rows(values.items(), ["Indicador", "Valor"])


def show_rows(rows, columns=None):
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(rows, columns=columns))

REGION = "us-east-1"
BRONZE_BUCKET = "american-tower-dev-bronze-b7f9699786f4c119558b9d30f9"
OPERATIONS_BUCKET = "american-tower-dev-operations-1dd72256126674a5893dfca3df"

FIRST_PERIOD = (2019, 1)
LAST_PERIOD = (2026, 1)
TEMP_DIR = "/mnt"

# Opcional. Cambia un origen por la URL HTTPS o la ruta absoluta del original en EMR.
# Ejemplo: SOURCE_OVERRIDES = {"inei_distrito": "/home/hadoop/Distrito.rar"}
SOURCE_OVERRIDES = {}
show_values({"Región": REGION, "Bucket Bronze": BRONZE_BUCKET, "Bucket Operations": OPERATIONS_BUCKET,
             "Primer trimestre": f"{FIRST_PERIOD[0]}-T{FIRST_PERIOD[1]}",
             "Último trimestre": f"{LAST_PERIOD[0]}-T{LAST_PERIOD[1]}", "Temporal": TEMP_DIR})

,Indicador,Valor
0,Región,us-east-1
1,Bucket Bronze,american-tower-dev-bronze-b7f9699786f4c119558b9d30f9
2,Bucket Operations,american-tower-dev-operations-1dd72256126674a5893dfca3df
3,Primer trimestre,2019-T1
4,Último trimestre,2026-T1
5,Temporal,/mnt


## 2. Conexión a S3 y corrida de ingesta

In [2]:
import json
import uuid
from datetime import datetime, timezone

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client("s3", region_name=REGION, config=Config(retries={"mode": "standard", "max_attempts": 5}))


def publish_manifest(run_id, rows):
    key = f"run_id={run_id}/bronze/_SUCCESS.json"
    manifest = {
        "schema": "bronze-autonomo-v1",
        "complete": True,
        "run_id": run_id,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "objects": rows,
        "verified_bytes": sum(item["bytes"] for item in rows),
    }
    s3.put_object(
        Bucket=OPERATIONS_BUCKET, Key=key, IfNoneMatch="*", ContentType="application/json",
        Body=json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8"),
    )
    return {
        "manifest_uri": f"s3://{OPERATIONS_BUCKET}/{key}",
        "run_id": run_id,
        "objects": len(rows),
        "verified_bytes": manifest["verified_bytes"],
    }

for bucket in (BRONZE_BUCKET, OPERATIONS_BUCKET):
    try:
        s3.head_bucket(Bucket=bucket)
    except ClientError as error:
        raise RuntimeError(f"El rol de EMR no pudo acceder al bucket {bucket}: "
                           f"{error.response['Error']['Code']}.") from error

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:10]
show_values({"Bronze accesible": f"s3://{BRONZE_BUCKET}/run_id={run_id}/",
             "Operations accesible": OPERATIONS_BUCKET, "ID de corrida": run_id})

/opt/american-tower/venv/lib64/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Indicador,Valor
0,Bronze accesible,s3://american-tower-dev-bronze-b7f9699786f4c119558b9d30f9/run_id=20261125T071603Z-190d23d89e/
1,Operations accesible,american-tower-dev-operations-1dd72256126674a5893dfca3df
2,ID de corrida,20261125T071603Z-190d23d89e


## 3. Inventario de fuentes y períodos

In [3]:
from pathlib import Path
from urllib.parse import urlsplit

OOKLA_ROOT = "https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile"
INEI_ROOT = "https://ide.inei.gob.pe/files"
OSIPTEL_COPY = (
    "https://raw.githubusercontent.com/devEddu17x/american-tower-data-process-pipeline/"
    "89e1e207a553e8885c557d2bc859e434ebbf8fcb/datasets/osiptel"
)

# Los hashes fijan la versión de los dos archivos OSIPTEL utilizados por el proyecto.
INSTITUTIONAL = {
    "osiptel_cobertura": (
        "cobertura_osiptel.csv",
        ("https://www.datosabiertos.gob.pe/sites/default/files/Cobertura%20m%C3%B3vil%20por%20empresa%20operadora.csv",
         f"{OSIPTEL_COPY}/archivo_cobertura.csv"),
        "0e2c5336ca94ad9ab2e94dd1ae2d7a7a487ffda47c54e93267db9f0df575f2f0",
    ),
    "osiptel_estaciones": (
        "osiptel_estaciones.xlsx",
        ("https://osiptelgobpe.sharepoint.com/sites/RepositoriodeDatosAbiertosdelOSIPTEL/"
         "_layouts/15/download.aspx?UniqueId=46f3863d%2D003e%2D461e%2D9979%2D4a28383fd5a0",
         f"{OSIPTEL_COPY}/archivo_estaciones.xlsx"),
        "8d4e62daf1088a398ec608366f48559dac763141b93cf9530513466b3ec3bd82",
    ),
    "inei_distrito": ("Distrito.rar", (f"{INEI_ROOT}/Distrito.rar",), None),
    "inei_provincia": ("Provincia.rar", (f"{INEI_ROOT}/Provincia.rar",), None),
    "inei_departamento": ("Departamento.rar", (f"{INEI_ROOT}/Departamento.rar",), None),
    "worldpop": (
        "per_ppp_2020.tif",
        ("https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/PER/per_ppp_2020.tif",),
        None,
    ),
}


def source_inventory():
    unknown = SOURCE_OVERRIDES.keys() - INSTITUTIONAL.keys()
    if unknown:
        raise ValueError(f"Fuentes desconocidas en SOURCE_OVERRIDES: {', '.join(sorted(unknown))}.")
    first_year, first_q = FIRST_PERIOD
    last_year, last_q = LAST_PERIOD
    if not (1 <= first_q <= 4 and 1 <= last_q <= 4):
        raise ValueError("Los trimestres deben estar entre 1 y 4.")
    first = first_year * 4 + first_q - 1
    last = last_year * 4 + last_q - 1
    if first > last:
        raise ValueError("FIRST_PERIOD debe preceder o coincidir con LAST_PERIOD.")

    inventory = []
    # Los archivos institucionales se comprueban antes de los Parquet más pesados.
    for label, (name, uris, expected_sha) in INSTITUTIONAL.items():
        source = "inei" if label.startswith("inei_") else label
        if label in SOURCE_OVERRIDES:
            uri = SOURCE_OVERRIDES[label]
            parsed = urlsplit(uri)
            if (parsed.scheme == "https" and not parsed.netloc) or (
                parsed.scheme != "https" and (parsed.scheme or not Path(uri).is_absolute())
            ):
                raise ValueError(f"{label}: indicar una URL HTTPS o una ruta local absoluta en EMR.")
            if source == "inei":
                extension = Path(parsed.path).suffix.lower()
                if extension not in {".rar", ".zip", ".gpkg"}:
                    raise ValueError(f"{label}: se requiere un archivo RAR, ZIP o GeoPackage.")
                name = Path(name).with_suffix(extension).name
            uris = (uri,)
        inventory.append((label, source, name, uris, expected_sha))

    for period in range(first, last + 1):
        year, quarter0 = divmod(period, 4)
        quarter = quarter0 + 1
        name = f"{year}-{quarter0 * 3 + 1:02d}-01_performance_mobile_tiles.parquet"
        url = f"{OOKLA_ROOT}/year={year}/quarter={quarter}/{name}"
        inventory.append((f"Ookla {year}-T{quarter}", "ookla", name, (url,), None))
    return inventory
inventory = source_inventory()
institutional = [entry for entry in inventory if entry[1] != "ookla"]
mobile = [entry for entry in inventory if entry[1] == "ookla"]
show_rows([{"fuente": label, "objeto": name, "origen": uris[0], "origen_alterno": len(uris) > 1}
           for label, _, name, uris, _ in institutional])
show_values({"Parquet trimestrales": len(mobile), "Primer parquet": mobile[0][2],
             "Último parquet": mobile[-1][2], "Total de objetos": len(inventory)})

,fuente,objeto,origen,origen_alterno
0,osiptel_cobertura,cobertura_osiptel.csv,https://www.datosabiertos.gob.pe/sites/default/files/Cobertura%20m%C3%B3vil%20por%20empresa%20operadora.csv,True
1,osiptel_estaciones,osiptel_estaciones.xlsx,https://osiptelgobpe.sharepoint.com/sites/RepositoriodeDatosAbiertosdelOSIPTEL/_layouts/15/download.aspx?UniqueId=46f3863d%2D003e%2D461e%2D9979%2D4a28383fd5a0,True
2,inei_distrito,Distrito.rar,https://ide.inei.gob.pe/files/Distrito.rar,False
3,inei_provincia,Provincia.rar,https://ide.inei.gob.pe/files/Provincia.rar,False
4,inei_departamento,Departamento.rar,https://ide.inei.gob.pe/files/Departamento.rar,False
5,worldpop,per_ppp_2020.tif,https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/PER/per_ppp_2020.tif,False


,Indicador,Valor
0,Parquet trimestrales,29
1,Primer parquet,2019-01-01_performance_mobile_tiles.parquet
2,Último parquet,2026-01-01_performance_mobile_tiles.parquet
3,Total de objetos,35


## 4. Ingesta de fuentes institucionales

In [4]:
import base64
import hashlib
import shutil
import time
from contextlib import contextmanager
from http.client import HTTPException
from tempfile import TemporaryDirectory
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from zipfile import BadZipFile, ZipFile

BLOCK_SIZE = 8 * 1024**2


@contextmanager
def open_source(uri):
    if uri.startswith("https://"):
        request = Request(uri, headers={"User-Agent": "AmericanTower-EMR-ingesta/1.0", "Accept-Encoding": "identity"})
        with urlopen(request, timeout=90) as response:
            if urlsplit(response.geturl()).scheme != "https":
                raise ValueError("El proveedor redirigió la descarga fuera de HTTPS.")
            length = response.headers.get("Content-Length")
            yield response, int(length) if length else None, response.geturl(), response.headers.get("ETag")
    else:
        path = Path(uri)
        with path.open("rb") as stream:
            yield stream, path.stat().st_size, path.as_uri(), None


def check_container(path):
    with path.open("rb") as stream:
        magic = stream.read(16)
        extension = path.suffix
        if extension == ".parquet":
            stream.seek(-4, 2)
            valid = magic.startswith(b"PAR1") and stream.read(4) == b"PAR1"
        elif extension == ".rar":
            valid = magic.startswith(b"Rar!\x1a\x07")
        elif extension == ".gpkg":
            valid = magic.startswith(b"SQLite format 3")
        elif extension == ".tif":
            valid = magic[:4] in {b"II*\x00", b"MM\x00*", b"II+\x00", b"MM\x00+"}
        elif extension in {".xlsx", ".zip"}:
            with ZipFile(path) as archive:
                names = archive.namelist()
                valid = ("xl/workbook.xml" in names if extension == ".xlsx"
                         else any(name.lower().endswith(".gpkg") for name in names))
        else:
            stream.seek(0)
            header = stream.readline(2048).decode("cp1252").upper()
            valid = all(name in header for name in ("UBIGEO_DISTRITO", "PERIODO", "EMPRESA_OPERADORA"))
    if not valid:
        raise ValueError(f"La descarga no contiene el formato esperado de {path.name}.")


def download(uri, path, expected_sha):
    digest = hashlib.sha256()
    size = 0
    with open_source(uri) as (stream, expected, actual_uri, etag), path.open("wb") as target:
        if expected is not None and expected > shutil.disk_usage(path.parent).free:
            raise ValueError(f"Espacio temporal insuficiente para {path.name} en {path.parent}.")
        for chunk in iter(lambda: stream.read(BLOCK_SIZE), b""):
            target.write(chunk)
            digest.update(chunk)
            size += len(chunk)
    if not size or (expected is not None and size != expected):
        raise OSError(f"Descarga incompleta de {path.name}: {size} bytes, esperados {expected}.")
    if expected_sha and digest.hexdigest() != expected_sha:
        raise ValueError(f"El archivo {path.name} difiere de la versión SHA-256 fijada para el proyecto.")
    check_container(path)
    return size, digest.hexdigest(), actual_uri, etag


def fetch_source(label, name, uris, expected_sha, directory):
    path = directory / name
    failures = []
    for uri in uris:
        for attempt in range(1, 4):
            retry = False
            try:
                size, digest, actual_uri, etag = download(uri, path, expected_sha)
                return path, size, digest, actual_uri, etag
            except HTTPError as error:
                failures.append(f"{uri} → HTTP {error.code}")
                retry = error.code in {429, 500, 502, 503, 504}
                error.close()
            except (URLError, OSError, HTTPException) as error:
                failures.append(f"{uri} → {error}")
                retry = uri.startswith("https://")
            except (ValueError, BadZipFile) as error:
                failures.append(f"{uri} → {error}")
            path.unlink(missing_ok=True)
            if not retry or attempt == 3:
                break
            time.sleep(2 ** (attempt - 1))
    hint = (f" Si el portal no responde, usa SOURCE_OVERRIDES[{label!r}] con el original."
            if label in INSTITUTIONAL else "")
    raise RuntimeError(f"No se pudo obtener {label}.{hint}\n" + "\n".join(dict.fromkeys(failures)))


def upload_original(run_id, source, name, downloaded):
    path, size, sha, origin, etag = downloaded
    key = f"run_id={run_id}/{source}/{name}"
    checksum = base64.b64encode(bytes.fromhex(sha)).decode("ascii")
    with path.open("rb") as stream:
        response = s3.put_object(
            Bucket=BRONZE_BUCKET, Key=key, Body=stream, ContentLength=size,
            IfNoneMatch="*", ChecksumSHA256=checksum, Metadata={"sha256": sha},
        )
    head = s3.head_object(Bucket=BRONZE_BUCKET, Key=key, ChecksumMode="ENABLED")
    if (response.get("ChecksumSHA256") != checksum or head.get("ChecksumSHA256") != checksum
            or head["ContentLength"] != size or head["Metadata"].get("sha256") != sha):
        raise ValueError(f"La verificación en S3 falló para {name}.")
    path.unlink()
    return {"source": source, "bucket": BRONZE_BUCKET, "key": key, "bytes": size, "sha256": sha,
            "status": "verified", "version_id": head.get("VersionId"), "source_uri": origin,
            "source_etag": etag, "obtained_at_utc": datetime.now(timezone.utc).isoformat()}



def ingest_group(inventory, title):
    rows = []
    with TemporaryDirectory(prefix="bronze-", dir=TEMP_DIR) as folder:
        for index, (label, source, name, uris, expected_sha) in enumerate(inventory, 1):
            print(f"[{index}/{len(inventory)}] {label} · {name}", flush=True)
            downloaded = fetch_source(label, name, uris, expected_sha, Path(folder))
            rows.append(upload_original(run_id, source, name, downloaded))
    show_rows([{"fuente": item["source"], "archivo": Path(item["key"]).name, "bytes": item["bytes"],
                "estado": item["status"], "sha256": item["sha256"]} for item in rows])
    print(f"{title}: {len(rows)} objetos verificados.", flush=True)
    return rows

institutional_receipts = ingest_group(institutional, "Fuentes institucionales")

[1/6] osiptel_cobertura · cobertura_osiptel.csv
[2/6] osiptel_estaciones · osiptel_estaciones.xlsx
[3/6] inei_distrito · Distrito.rar
[4/6] inei_provincia · Provincia.rar
[5/6] inei_departamento · Departamento.rar
[6/6] worldpop · per_ppp_2020.tif


,fuente,archivo,bytes,estado,sha256
0,osiptel_cobertura,cobertura_osiptel.csv,7590550,verified,0e2c5336ca94ad9ab2e94dd1ae2d7a7a487ffda47c54e93267db9f0df575f2f0
1,osiptel_estaciones,osiptel_estaciones.xlsx,740518,verified,8d4e62daf1088a398ec608366f48559dac763141b93cf9530513466b3ec3bd82
2,inei,Distrito.rar,13053124,verified,ae22428029000d801b3d68afd0914ba94ca40a789a9b56231f6131a75abb9654
3,inei,Provincia.rar,5591257,verified,e267ab37ecbce3ed351e983a498ead730eeba08158238bcb1c192442402ebd8d
4,inei,Departamento.rar,2763519,verified,c6c2b3075229e1b4bfb33e9bfe4979d0a71dbf452f4b04b7c831928bdfc574da
5,worldpop,per_ppp_2020.tif,704980975,verified,dd360290d7a92814198fcaca6fa74e2d05a1edb692b4f249d5e1100b48ec95fa


Fuentes institucionales: 6 objetos verificados.


## 5. Ingesta de las teselas móviles de Ookla

In [5]:
mobile_receipts = ingest_group(mobile, "Parquet Ookla")

[1/29] Ookla 2019-T1 · 2019-01-01_performance_mobile_tiles.parquet
[2/29] Ookla 2019-T2 · 2019-04-01_performance_mobile_tiles.parquet
[3/29] Ookla 2019-T3 · 2019-07-01_performance_mobile_tiles.parquet
[4/29] Ookla 2019-T4 · 2019-10-01_performance_mobile_tiles.parquet
[5/29] Ookla 2020-T1 · 2020-01-01_performance_mobile_tiles.parquet
[6/29] Ookla 2020-T2 · 2020-04-01_performance_mobile_tiles.parquet
[7/29] Ookla 2020-T3 · 2020-07-01_performance_mobile_tiles.parquet
[8/29] Ookla 2020-T4 · 2020-10-01_performance_mobile_tiles.parquet
[9/29] Ookla 2021-T1 · 2021-01-01_performance_mobile_tiles.parquet
[10/29] Ookla 2021-T2 · 2021-04-01_performance_mobile_tiles.parquet
[11/29] Ookla 2021-T3 · 2021-07-01_performance_mobile_tiles.parquet
[12/29] Ookla 2021-T4 · 2021-10-01_performance_mobile_tiles.parquet
[13/29] Ookla 2022-T1 · 2022-01-01_performance_mobile_tiles.parquet
[14/29] Ookla 2022-T2 · 2022-04-01_performance_mobile_tiles.parquet
[15/29] Ookla 2022-T3 · 2022-07-01_performance_mobile_til

,fuente,archivo,bytes,estado,sha256
0,ookla,2019-01-01_performance_mobile_tiles.parquet,148611294,verified,b15399ef178ad2e266e18efee4e7bc40aedafd47669deb8b1f015300ef9b71b7
1,ookla,2019-04-01_performance_mobile_tiles.parquet,154034407,verified,a57f3bc407a1370b472f67af5b4dad185a9dc79406843d7f40b60dac7ea4e9a3
2,ookla,2019-07-01_performance_mobile_tiles.parquet,184820230,verified,cabe67da05295fd3a9035b71a0d9af4f3a5b76d59cf4fb04db86419ee6ce2fc3
3,ookla,2019-10-01_performance_mobile_tiles.parquet,175335646,verified,a6a948618e1aeedfbab42498dba428735cad504c0ebeaf5ea942feb7f4c0e31e
4,ookla,2020-01-01_performance_mobile_tiles.parquet,189988511,verified,67e8bd351514e460d95f25105361ea451b5ec6edecd167be514fa1a49c4dc77b
5,ookla,2020-04-01_performance_mobile_tiles.parquet,199312810,verified,1b4071e11eaca7b056f10eee5a54ac6abe11ef653eb9d6f56d07b5d9b305ebf3
6,ookla,2020-07-01_performance_mobile_tiles.parquet,213172798,verified,4b2d400b3b2329e3b4bc1680a0a35e89a93a9f7c6197f1fb464fad9d5bbe8a50
7,ookla,2020-10-01_performance_mobile_tiles.parquet,206365569,verified,06ba45745c3ae0c9c9bb664b69426be0ec0becb19a12e138f4067bc887ec8687
8,ookla,2021-01-01_performance_mobile_tiles.parquet,205713556,verified,a62ebb1d670e8aa8fc0cdb86d05aaaeef8391158ac1590a4c6466b776a054501
9,ookla,2021-04-01_performance_mobile_tiles.parquet,211627891,verified,670c8ddca52fe883f539050552f54535f730d1ce821c44c50c13e8a9cda56fb2


Parquet Ookla: 29 objetos verificados.


## 6. Conciliación y publicación del manifiesto Bronze

In [6]:
rows = institutional_receipts + mobile_receipts
expected = {(source, name) for _, source, name, _, _ in inventory}
actual = {(row["source"], Path(row["key"]).name) for row in rows}
if (actual != expected or len(rows) != len(inventory)
        or any(row["status"] != "verified" for row in rows)):
    raise ValueError("El inventario esperado y los objetos verificados en Bronze no coinciden.")
bronze_evidence = pd.DataFrame(rows).groupby("source", as_index=False).agg(
    objetos=("key", "count"), bytes=("bytes", "sum")
)
display(bronze_evidence)
resultado = publish_manifest(run_id, rows)
show_values(resultado)
print(f"BRONZE_RUN_URI = {resultado['manifest_uri']!r}")

,source,objetos,bytes
0,inei,3,21407900
1,ookla,29,5676137401
2,osiptel_cobertura,1,7590550
3,osiptel_estaciones,1,740518
4,worldpop,1,704980975


,Indicador,Valor
0,manifest_uri,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T071603Z-190d23d89e/bronze/_SUCCESS.json
1,run_id,20261125T071603Z-190d23d89e
2,objects,35
3,verified_bytes,6410857344


BRONZE_RUN_URI = 's3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T071603Z-190d23d89e/bronze/_SUCCESS.json'
